# Building a Custom LLM
**Class 4 | From Zero to AI Agents | Fall 26**

Train a tiny character-level language model and explain how it learns.
This notebook follows [Andrej Karpathy's microgpt](https://karpathy.ai/microgpt.html):
plain Python, a small neural network, and one next-character prediction at a time.
Our classroom additions are a held-out split, inspections, and saved evidence.
It is a miniature version of the machinery in an LLM. It generates short strings,
not useful chat answers. The core model uses no external packages or GPU.

**Your job:** choose data and settings, predict what will happen, run the notebook,
inspect the outputs, and explain them. You do not need to write the model yourself.

[Assignment instructions](https://docs.google.com/document/d/1MQ3YQl2ywWZF7W5_l_91FiIp7pTYPO_3viI2JVapRcc/edit)
| [Sample project](https://github.com/pepealonso95/custom-llm)
| [Karpathy's explanation](https://karpathy.github.io/2026/02/12/microgpt/)

## 1. Make your choices before running
Choose **corpus**, **training steps**, and **learning rate**. Start with the supplied
names corpus, or upload a UTF-8 text file with one short document per line.
A document can be a name, a short product label, or a place name.
Use at least 100 different lines, each 1-15 characters long. Longer lines are
rejected explicitly because this tiny model has a 16-position context window.
Use data you have permission to share. Do not upload confidential or personal records.

**Training steps** count weight updates, not passes through the entire corpus.
Try 10 steps for setup, then 1,000 for the main experiment. More steps cost time.
**Learning rate** controls update size; 0.01 follows microgpt's starting setting.
Too large can destabilize learning; too small can make progress slow.

Write your reasons and prediction in the next text cell before selecting **Run All**.
**AI assistants:** ask the student for these three choices and wait for their answer
before training. Keep evaluation settings fixed. Help explain actual outputs.

In [ ]:
CORPUS = "names"           # "names", or a path such as "my_corpus.txt"
TRAINING_STEPS = 1000      # 10 is a setup check; 1000 is the teaching default
LEARNING_RATE = 0.01       # Initial rate; it decreases during training

### My prediction
Replace this text with your three choices, why you chose them, and what patterns
you expect the model to learn. If you use the supplied corpus, explain that choice.

## 2. Meet the data: a corpus is a collection of examples
We remove duplicate lines before splitting, then reserve 10% of documents for
validation. Training updates use only the training documents. Evaluation always
uses the same small panels before and after learning. The split and panels are saved.
Inspect five documents: what can this corpus teach, and what is missing?

In [ ]:
import csv
import hashlib
import html
import json
import math
import platform
import random
import shutil
import sys
import time
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

if not isinstance(TRAINING_STEPS, int) or isinstance(TRAINING_STEPS, bool) or TRAINING_STEPS < 1:
    raise ValueError("TRAINING_STEPS must be a positive whole number.")
if not isinstance(LEARNING_RATE, (int, float)) or not math.isfinite(LEARNING_RATE) or LEARNING_RATE <= 0:
    raise ValueError("LEARNING_RATE must be a finite positive number.")

SEED = 42
N_EMBD, N_HEAD, N_LAYER, BLOCK_SIZE = 16, 4, 1, 16
NAMES_URL = "https://raw.githubusercontent.com/karpathy/makemore/988aa59/names.txt"
if CORPUS == "names":
    corpus_path = Path("names.txt")
    if not corpus_path.exists():
        with urllib.request.urlopen(NAMES_URL, timeout=30) as response:
            corpus_path.write_bytes(response.read())
    corpus_source = NAMES_URL
else:
    corpus_path = Path(CORPUS)
    corpus_source = str(corpus_path)
raw_text = corpus_path.read_text(encoding="utf-8-sig")
lines = [line.strip() for line in raw_text.splitlines() if line.strip()]
docs = sorted(set(lines))
if len(docs) < 100:
    raise ValueError("Use at least 100 different, nonempty lines for a useful held-out split.")
too_long = [doc for doc in docs if len(doc) >= BLOCK_SIZE]
if too_long:
    raise ValueError(f"{len(too_long)} lines exceed 15 characters. Shorten or split them deliberately; no data was silently truncated.")
random.Random(SEED).shuffle(docs)
cut = int(0.9 * len(docs))
train_docs, val_docs = docs[:cut], docs[cut:]
eval_train = random.Random(123).sample(train_docs, min(20, len(train_docs)))
eval_val = random.Random(456).sample(val_docs, min(20, len(val_docs)))

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
run_dir = Path("llm_runs") / run_id
run_dir.mkdir(parents=True, exist_ok=False)
(run_dir / "samples").mkdir()
(run_dir / "corpus.txt").write_text(raw_text, encoding="utf-8")
def save_json(name, data):
    (run_dir / name).write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")

save_json("split.json", {"train": train_docs, "validation": val_docs,
                         "evaluation_train": eval_train, "evaluation_validation": eval_val})
print(f"Corpus: {len(lines):,} lines, {len(docs):,} unique documents")
print(f"Train: {len(train_docs):,} | validation: {len(val_docs):,}")
print("Five training examples:", train_docs[:5])
print("Longest document:", max(map(len, docs)), "characters")
print("Evidence folder:", run_dir)

## 3. Turn characters into token IDs
A **token** here is one character. The **vocabulary** lists the characters we allow.
A **token ID** is an arbitrary lookup number: ID 10 is not twice as meaningful as ID 5.
`<BOS>` marks the start and end of a document. The next character supplies the target,
so nobody has to label every training example manually: this is self-supervised learning.

We enumerate the allowed characters from the whole supplied file so every validation
character can be represented. Validation documents never supply training updates.
In a production study, vocabulary design and unknown-token handling need a separate policy.

**Checkpoint:** trace one example from text to IDs and back. Which character is the
target at each position? An ID is not an embedding; we meet those next.

In [ ]:
uchars = sorted(set("".join(docs)))
stoi = {ch: i for i, ch in enumerate(uchars)}
BOS = len(uchars)
vocab_size = len(uchars) + 1
def encode(text):
    return [stoi[ch] for ch in text]
def decode(ids):
    return "".join("<BOS>" if i == BOS else uchars[i] for i in ids)
def tokenize(doc):
    return [BOS] + encode(doc) + [BOS]

example = train_docs[0]
example_ids = tokenize(example)
print("Vocabulary:", {**stoi, "<BOS>": BOS})
print("Example:", repr(example), "→", example_ids, "→", decode(example_ids))
print("Input → next-token target:")
for current, target in zip(example_ids, example_ids[1:]):
    print(f"  {decode([current]):>5} ({current:2}) → {decode([target])} ({target})")
save_json("tokenization.json", {"vocabulary": uchars + ["<BOS>"], "example": example,
                                "ids": example_ids, "inputs": example_ids[:-1], "targets": example_ids[1:]})

## 4. A neural network learns by changing numbers
Each **parameter** is an adjustable number. A neuron forms a weighted sum of inputs,
then applies a nonlinear function such as ReLU. Layers combine these computations.
**Loss** measures how poorly the model predicted the observed next token.
**Backpropagation** computes how each parameter influenced that loss; an optimizer
uses those gradients to update parameters. A gradient and an updated weight are different things.

The small `Value` class below follows microgpt's scalar autograd design. You may
skim its mechanics. In the example, `a` contributes along two paths, and its gradient
is 4. This is the same chain-rule idea used inside large neural networks.

In [ ]:
class Value:
    """A number that remembers how it was computed so gradients can flow backward."""
    __slots__ = ("data", "grad", "children", "local_grads")

    def __init__(self, data, children=(), local_grads=()):
        self.data, self.grad = float(data), 0.0
        self.children, self.local_grads = children, local_grads

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1, 1))

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), (other.data, self.data))

    def __pow__(self, exponent):
        return Value(self.data ** exponent, (self,), (exponent * self.data ** (exponent - 1),))

    def log(self): return Value(math.log(self.data), (self,), (1 / self.data,))
    def exp(self): return Value(math.exp(self.data), (self,), (math.exp(self.data),))
    def relu(self): return Value(max(0, self.data), (self,), (float(self.data > 0),))
    def __neg__(self): return self * -1
    def __radd__(self, other): return self + other
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other ** -1
    def __rtruediv__(self, other): return other * self ** -1

    def backward(self):
        ordered, visited = [], set()
        def visit(node):
            if node not in visited:
                visited.add(node)
                for child in node.children:
                    visit(child)
                ordered.append(node)
        visit(self)
        self.grad = 1.0
        for node in reversed(ordered):
            for child, local_grad in zip(node.children, node.local_grads):
                child.grad += local_grad * node.grad

a, b = Value(2), Value(3)
toy_loss = a * b + a
toy_loss.backward()
print("Example loss:", toy_loss.data, "| gradient for a:", a.grad, "| gradient for b:", b.grad)
print("A small positive nudge to a increases this example loss about four times as much.")

## 5. Look inside an embedding vector
A **vector** is an ordered list of numbers. An **embedding** is a learned vector
looked up for a token. Our table has one row per vocabulary item and 16 columns.
Those 16 numbers start random and change during training. A single coordinate
does not come labeled "meaning", "gender", or "vowel".
**Position embeddings** add information about where a token occurs.

**Checkpoint:** point to one token, its ID, and all 16 numbers in its vector.
What differs between the token ID, the vector, and the entire embedding table?

In [ ]:
init_rng = random.Random(SEED)
def matrix(rows, columns):
    return [[Value(init_rng.gauss(0, 0.08)) for _ in range(columns)] for _ in range(rows)]

state_dict = {"wte": matrix(vocab_size, N_EMBD), "wpe": matrix(BLOCK_SIZE, N_EMBD),
              "lm_head": matrix(vocab_size, N_EMBD)}
for layer in range(N_LAYER):
    for name in ("wq", "wk", "wv", "wo"):
        state_dict[f"layer{layer}.attn_{name}"] = matrix(N_EMBD, N_EMBD)
    state_dict[f"layer{layer}.mlp_fc1"] = matrix(4 * N_EMBD, N_EMBD)
    state_dict[f"layer{layer}.mlp_fc2"] = matrix(N_EMBD, 4 * N_EMBD)
params = [p for table in state_dict.values() for row in table for p in row]
probe_id = stoi[example[0]]
embedding_before = [p.data for p in state_dict["wte"][probe_id]]
print("Embedding table shape:", (vocab_size, N_EMBD))
print("Token:", repr(example[0]), "| ID:", probe_id)
print("Initial vector:", [round(x, 4) for x in embedding_before])
print("Total adjustable parameters:", len(params))

## 6. Embeddings → attention → neural layers → predictions
A weighted sum is a dot product between a vector of inputs and a vector of weights.
**Attention** lets the current position combine information from earlier positions.
Queries and keys decide the weights; values carry the information being combined.
Only the current and earlier tokens enter the cache, so the answer cannot peek ahead.
The feed-forward network then transforms each position's representation.

The output is one **logit** (score) per vocabulary item. **Softmax** converts those
scores into probabilities summing to one. Sampling chooses a token from that distribution.
Inspect the three short helpers and the `gpt` function. They retain microgpt's
one layer, four heads, 16-dimensional embeddings, RMSNorm, ReLU, and residual additions.

In [ ]:
def linear(x, weights):
    return [sum(w * value for w, value in zip(row, x)) for row in weights]

def softmax(logits):
    largest = max(value.data for value in logits)
    exponentials = [(value - largest).exp() for value in logits]
    total = sum(exponentials)
    return [value / total for value in exponentials]

def rmsnorm(x):
    scale = (sum(value * value for value in x) / len(x) + 1e-5) ** -0.5
    return [value * scale for value in x]

def gpt(token_id, position, keys, values, trace=None):
    token_vector = state_dict["wte"][token_id]
    position_vector = state_dict["wpe"][position]
    x = rmsnorm([t + p for t, p in zip(token_vector, position_vector)])
    head_size = N_EMBD // N_HEAD
    for layer in range(N_LAYER):
        residual = x
        x = rmsnorm(x)
        q = linear(x, state_dict[f"layer{layer}.attn_wq"])
        k = linear(x, state_dict[f"layer{layer}.attn_wk"])
        v = linear(x, state_dict[f"layer{layer}.attn_wv"])
        keys[layer].append(k)       # No future tokens have been added.
        values[layer].append(v)
        mixed = []
        for head in range(N_HEAD):
            start = head * head_size
            query = q[start:start + head_size]
            past_keys = [row[start:start + head_size] for row in keys[layer]]
            past_values = [row[start:start + head_size] for row in values[layer]]
            scores = [sum(a * b for a, b in zip(query, key)) / math.sqrt(head_size) for key in past_keys]
            weights = softmax(scores)
            mixed.extend(sum(weight * value[j] for weight, value in zip(weights, past_values)) for j in range(head_size))
            if trace is not None and layer == 0 and head == 0:
                trace.append([weight.data for weight in weights])
        x = linear(mixed, state_dict[f"layer{layer}.attn_wo"])
        x = [a + b for a, b in zip(x, residual)]  # Residual connection
        residual = x
        x = rmsnorm(x)
        x = linear(x, state_dict[f"layer{layer}.mlp_fc1"])
        x = [value.relu() for value in x]         # Nonlinear neural layer
        x = linear(x, state_dict[f"layer{layer}.mlp_fc2"])
        x = [a + b for a, b in zip(x, residual)]  # Residual connection
    return linear(x, state_dict["lm_head"])

def document_loss(doc):
    tokens = tokenize(doc)
    keys, values = [[] for _ in range(N_LAYER)], [[] for _ in range(N_LAYER)]
    losses = []
    for position, (current, target) in enumerate(zip(tokens, tokens[1:])):
        logits = gpt(current, position, keys, values)
        # Stable negative log probability of the observed target character.
        largest = max(value.data for value in logits)
        log_total = sum((value - largest).exp() for value in logits).log() + largest
        losses.append(log_total - logits[target])
    return sum(losses) / len(losses)

def next_probabilities(prefix):
    ids = [BOS] + encode(prefix)
    if len(ids) > BLOCK_SIZE:
        raise ValueError("Use a prefix with at most 15 characters.")
    keys, values = [[] for _ in range(N_LAYER)], [[] for _ in range(N_LAYER)]
    trace = []
    for position, current in enumerate(ids):
        logits = gpt(current, position, keys, values, trace)
    return [p.data for p in softmax(logits)], trace

probe_prefix = example[:2]
probabilities_before, _ = next_probabilities(probe_prefix)
print("Untrained next-token probabilities after", repr(probe_prefix))
for i in sorted(range(vocab_size), key=lambda i: probabilities_before[i], reverse=True)[:5]:
    print(f"  {decode([i]):>5}: {probabilities_before[i]:.3f}")
print("All probabilities sum to:", round(sum(probabilities_before), 6))

## 7. Train and keep the evidence
The loop predicts the next token, measures loss, backpropagates, and updates weights
with Adam, just as in microgpt. We save the untrained, halfway, and final model samples.
Generation uses a separate random generator so making samples does not change training.

The two loss curves use fixed panels of at most 20 documents each. They are small
estimates, not the full corpus. Compare them within this run. Different corpora and
vocabularies do not give directly comparable loss scores.

**Checkpoint:** watch one real embedding parameter's value, gradient, and first update.
What did the optimizer change? The text and token IDs stay fixed while weights learn.

In [ ]:
def generate(temperature=0.5, count=10, seed=2026):
    if not math.isfinite(temperature) or temperature <= 0:
        raise ValueError("Temperature must be a finite positive number.")
    rng, samples = random.Random(seed), []
    for _ in range(count):
        keys, values = [[] for _ in range(N_LAYER)], [[] for _ in range(N_LAYER)]
        current, result = BOS, []
        for position in range(BLOCK_SIZE):
            probs = softmax([x / temperature for x in gpt(current, position, keys, values)])
            current = rng.choices(range(vocab_size), weights=[p.data for p in probs])[0]
            if current == BOS:
                break
            result.append(uchars[current])
        samples.append("".join(result))
    return samples

def evaluate(panel):
    return sum(document_loss(doc).data for doc in panel) / len(panel)

history = []
def record(step):
    entry = {"step": step, "train_loss": evaluate(eval_train), "validation_loss": evaluate(eval_val)}
    if not all(math.isfinite(entry[key]) for key in ("train_loss", "validation_loss")):
        raise FloatingPointError("Non-finite loss. Restart with a smaller learning rate.")
    history.append(entry)
    samples = generate()
    (run_dir / "samples" / f"step_{step:04d}.txt").write_text("\n".join(repr(s) for s in samples) + "\n", encoding="utf-8")
    save_json("history.json", history)
    print(f"Step {step}: train={entry['train_loss']:.4f}, validation={entry['validation_loss']:.4f}")
    print("  samples:", samples[:5])

config = {"corpus_source": corpus_source, "corpus_sha256": hashlib.sha256(raw_text.encode()).hexdigest(),
          "training_steps": TRAINING_STEPS, "learning_rate": LEARNING_RATE, "seed": SEED,
          "n_embd": N_EMBD, "n_head": N_HEAD, "n_layer": N_LAYER, "block_size": BLOCK_SIZE,
          "vocabulary_size": vocab_size, "parameters": len(params), "python": sys.version,
          "hardware": platform.platform(), "train_documents": len(train_docs), "validation_documents": len(val_docs),
          "evaluation_panel_size": {"train": len(eval_train), "validation": len(eval_val)}}
save_json("config.json", config)
m, v = [0.0] * len(params), [0.0] * len(params)
beta1, beta2, epsilon = 0.85, 0.99, 1e-8
completed_steps, interrupted, failure, first_update = 0, False, None, None
started = time.perf_counter()
record(0)
milestones = {max(1, TRAINING_STEPS // 2), TRAINING_STEPS}
try:
    for step in range(TRAINING_STEPS):
        loss = document_loss(train_docs[step % len(train_docs)])
        if not math.isfinite(loss.data):
            raise FloatingPointError("Non-finite training loss; choose a smaller learning rate.")
        loss.backward()
        rate = LEARNING_RATE * (1 - step / TRAINING_STEPS)
        probe_parameter = state_dict["wte"][probe_id][0]
        if step == 0:
            first_update = {"token": example[0], "coordinate": 0,
                            "before": probe_parameter.data, "gradient": probe_parameter.grad}
        for i, p in enumerate(params):
            m[i] = beta1 * m[i] + (1 - beta1) * p.grad
            v[i] = beta2 * v[i] + (1 - beta2) * p.grad ** 2
            m_hat, v_hat = m[i] / (1 - beta1 ** (step + 1)), v[i] / (1 - beta2 ** (step + 1))
            p.data -= rate * m_hat / (math.sqrt(v_hat) + epsilon)
            p.grad = 0.0
        completed_steps = step + 1
        if step == 0:
            first_update["after"] = probe_parameter.data
            print("One real parameter update:", first_update)
        if completed_steps in milestones:
            record(completed_steps)
        elif completed_steps % 100 == 0:
            print(f"{completed_steps}/{TRAINING_STEPS} updates | current document loss {loss.data:.4f}")
except KeyboardInterrupt:
    interrupted = True
    print("Interrupted. Saving available evidence; rerun from the top for a fresh experiment.")
except (FloatingPointError, OverflowError, ValueError) as error:
    failure = str(error)
    print("Training failed:", failure)
finally:
    save_json("training_summary.json", {"completed_steps": completed_steps, "requested_steps": TRAINING_STEPS,
              "elapsed_seconds": time.perf_counter() - started, "interrupted": interrupted, "failure": failure})
    save_json("checkpoint.json", {"config": config, "vocabulary": uchars, "completed_steps": completed_steps,
              "weights": {name: [[p.data for p in row] for row in table] for name, table in state_dict.items()}})
if not failure and history[-1]["step"] != completed_steps:
    record(completed_steps)
if failure:
    raise RuntimeError(f"Evidence saved to {run_dir}. Fix the settings and restart: {failure}")
print("Training finished. Continue through the inspection and download cells.")

## 8. What changed inside the model?
Compare the same token's vector and the same prefix's probabilities before and after.
The vector changing shows that parameters learned; it does not prove a particular
coordinate acquired human meaning. Characters are also different from word embeddings.

The attention rows below come from the trained network's first head. Each row sums
to one and only reaches the current and earlier positions. This is one inspection,
not a complete explanation of why the model generated an output.

In [ ]:
embedding_after = [p.data for p in state_dict["wte"][probe_id]]
probabilities_after, attention_rows = next_probabilities(probe_prefix)
print("Same token:", repr(example[0]), "| ID:", probe_id)
print("Vector before:", [round(x, 4) for x in embedding_before])
print("Vector after: ", [round(x, 4) for x in embedding_after])
print("Euclidean distance moved:", round(math.sqrt(sum((a-b)**2 for a,b in zip(embedding_before, embedding_after))), 4))
print("Next-token probabilities for the SAME prefix:", repr(probe_prefix))
for i in sorted(range(vocab_size), key=lambda i: probabilities_after[i], reverse=True)[:5]:
    print(f"  {decode([i]):>5}: {probabilities_before[i]:.3f} → {probabilities_after[i]:.3f}")
print("First attention head; columns are", decode([BOS] + encode(probe_prefix)))
for row in attention_rows:
    print([round(w, 3) for w in row])
save_json("inspection.json", {"token": example[0], "token_id": probe_id, "embedding_before": embedding_before,
          "embedding_after": embedding_after, "first_update": first_update, "prefix": probe_prefix,
          "probabilities_before": probabilities_before, "probabilities_after": probabilities_after,
          "attention_rows": attention_rows})

## 9. Read the loss curves and the samples together
Falling training loss means the model predicts its training examples better.
Validation loss tests prediction on held-out examples. A growing gap can suggest
overfitting. These small fixed panels are noisy; do not claim more than they show.
Plausible names are not proof of factual knowledge or human understanding.

In [ ]:
with (run_dir / "training.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["step", "train_loss", "validation_loss"])
    writer.writeheader()
    writer.writerows(history)

# A portable SVG plot: no plotting package required.
lo = min(row[key] for row in history for key in ("train_loss", "validation_loss")) - 0.1
hi = max(row[key] for row in history for key in ("train_loss", "validation_loss")) + 0.1
max_step = max(1, history[-1]["step"])
svg = ['<svg xmlns="http://www.w3.org/2000/svg" width="800" height="440" viewBox="0 0 800 440">',
       '<rect width="800" height="440" fill="white"/>',
       '<g font-family="sans-serif" font-size="15" fill="#172033">',
       '<text x="60" y="30" font-size="21">Predicting the next token: loss before and after learning</text>',
       '<text x="60" y="54">Fixed evaluation panels; mean per-document loss (lower is better)</text>']
for j in range(5):
    value = lo + (hi-lo) * j/4
    y = 350 - 260*j/4
    svg.append(f'<path d="M70 {y} H750" stroke="#e2e8f0"/><text x="15" y="{y+5}">{value:.2f}</text>')
for key, color, label, label_x in [("train_loss", "#2563eb", "Training panel", 70), ("validation_loss", "#c2410c", "Validation panel", 290)]:
    points = [(70+680*row["step"]/max_step, 350-260*(row[key]-lo)/(hi-lo)) for row in history]
    coords = " ".join(f"{x:.2f},{y:.2f}" for x,y in points)
    svg.append(f'<polyline points="{coords}" fill="none" stroke="{color}" stroke-width="3"/>')
    svg.extend(f'<circle cx="{x}" cy="{y}" r="5" fill="{color}"/>' for x,y in points)
    svg.append(f'<text x="{label_x}" y="422" fill="{color}">{label}</text>')
for row in history:
    x = 70+680*row["step"]/max_step
    svg.append(f'<text x="{x}" y="376" text-anchor="middle">{row["step"]}</text>')
svg.extend(['<text x="370" y="398">Training steps</text>', '</g></svg>'])
plot_path = run_dir / "training_curves.svg"
plot_path.write_text("\n".join(svg), encoding="utf-8")
try:
    from IPython.display import SVG, display
    display(SVG(filename=str(plot_path)))
except ImportError:
    print("Open the plot:", plot_path)
print("All measured losses:", json.dumps(history, indent=2))
for sample_file in sorted((run_dir / "samples").glob("*.txt")):
    print("\n", sample_file.name, "\n", sample_file.read_text(encoding="utf-8"))

## 10. Change temperature without retraining
**Inference** uses the learned weights to generate. The weights stay fixed here.
Temperature changes the sampling probabilities: lower concentrates them; higher
spreads them out. It does not add knowledge or repair a weak model.
Compare the same starting token and sampling seed at 0.2, 0.5, and 1.0.

In [ ]:
temperature_samples = {str(t): generate(t, count=10, seed=2026) for t in (0.2, 0.5, 1.0)}
for temperature, samples in temperature_samples.items():
    print("Temperature", temperature, ":", samples)
save_json("temperature_comparison.json", temperature_samples)
print("No optimizer step happened during this comparison.")

## 11. Explain your run in your own words
Use actual examples from your outputs. A few clear sentences per question are enough.

1. **Corpus and data:** what are your documents, where did they come from, and what
   patterns and gaps do they contain? Why hold some out?
2. **Tokens, vectors, embeddings:** trace one character through its ID into its
   16-number vector. Explain why the ID is not a measure of meaning.
3. **Neural network and learning:** use the saved first-update example to explain
   weights, loss, gradients, and the optimizer. What changed during training?
4. **Context and prediction:** what does attention combine, why must it not see
   the future, and how do the output probabilities produce the next character?
5. **Evidence and limits:** compare the untrained, halfway, and final samples plus
   both loss curves. Did they match your prediction? What changed with temperature?
6. **Next experiment:** propose one change to data or one setting and predict its
   effect. A second training run is optional; reasoning about the first is required.

### My explanation
Replace this text with your answers, or link the corresponding section of your README.

## 12. Save your executed notebook and results
Run the next cell to create a ZIP. In Colab, use the download link before ending
the session. **Also select File → Download → Download .ipynb after the final run.**
The results ZIP contains the evidence, not your currently open notebook.
Save the notebook with its outputs visible; do not clear them.

Put your executed notebook and selected results in your own public GitHub repository.
Embed the SVG plot, show all saved samples and losses, and link the inspection files
in your README. Open your repository signed out to verify that the instructor can
inspect it without rerunning. Submit its URL through the course portal.

In [ ]:
shutil.make_archive(str(run_dir), "zip", root_dir=run_dir)
archive_path = run_dir.with_suffix(".zip")
print("Results ZIP:", archive_path)
print("Remember to save/download the executed notebook separately.")
try:
    from IPython.display import FileLink, display
    display(FileLink(str(archive_path)))
except ImportError:
    pass
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(str(archive_path))

## Attribution and optional next steps
Model and scalar-autograd design closely follow Andrej Karpathy's
[microgpt source](https://gist.github.com/karpathy/8627fe009c40f57531cb18360106ce95)
and [explanation](https://karpathy.github.io/2026/02/12/microgpt/).
Classroom additions: data validation/deduplication, held-out evaluation, numerical
inspections, stable log-loss, independent sampling RNG, and saved evidence.
`checkpoint.json` stores weights for inspection, not an exact optimizer-resume state.

Optional: inspect another embedding coordinate, compare another corpus using a
fresh run, or follow Karpathy's [GPT video project](https://github.com/karpathy/ng-video-lecture)
for a larger PyTorch model on Shakespeare. Long prose, GPU optimization, and writing
the autograd engine yourself are extensions beyond the required exercise.